# Example 3: Evaluation and Visualization
# 학습된 모델을 평가하고 시각화하는 예제

이 노트북은 학습이 완료된 모델을 불러와서 평가하고 결과를 시각화하는 방법을 보여줍니다.


In [ ]:
import os
import sys

# Add parent directory to path
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

import yaml
import torch
import pandas as pd
from models import ViT50_3block
from dataloader import create_dataloaders
from utils import set_seed, evaluate_model, print_evaluation_summary, plot_results


## Step 1: Load Configuration
설정 파일을 불러옵니다.


In [ ]:
# Load configuration
config_path = './config_evaluation.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  - Checkpoint: {config['checkpoint']['load_path']}")
print(f"  - Dataset to evaluate: {config['evaluation']['dataset']}")
print(f"  - Batch Size: {config['dataloader']['batch_size']}")


## Step 2: Setup Environment


In [ ]:
# Set random seed
set_seed(config['seed'])

# Setup device
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories
os.makedirs(os.path.dirname(config['evaluation']['save_results_plot']), exist_ok=True)


## Step 3: Load Datasets
평가할 데이터셋을 선택합니다.


In [ ]:
print("Loading datasets...")
train_loader, val_loader, test_loader, lbl_min, lbl_max = create_dataloaders(
    train_path=config['data']['train_path'],
    val_path=config['data']['val_path'],
    test_path=config['data']['test_path'],
    batch_size=config['dataloader']['batch_size'],
    num_workers=config['dataloader']['num_workers'],
    pin_memory=config['dataloader']['pin_memory'],
    seed=config['seed']
)

# Select loader based on config
dataset_choice = config['evaluation']['dataset']
if dataset_choice == 'train':
    eval_loader = train_loader
    dataset_name = 'Training'
elif dataset_choice == 'val':
    eval_loader = val_loader
    dataset_name = 'Validation'
else:
    eval_loader = test_loader
    dataset_name = 'Test'

print(f"✓ Evaluating on {dataset_name} set ({len(eval_loader.dataset)} samples)")
print(f"✓ Label range: [{lbl_min:.4f}, {lbl_max:.4f}]")


## Step 4: Load Model and Checkpoint
모델을 생성하고 학습된 가중치를 불러옵니다.


In [ ]:
print("Creating model...")
model = ViT50_3block(
    img_size=config['model']['img_size'],
    patch_size=config['model']['patch_size'],
    embed_dim=config['model']['embed_dim'],
    depth=config['model']['depth'],
    num_heads=config['model']['num_heads'],
    mlp_dim=config['model']['mlp_dim'],
    num_classes=config['model']['num_classes']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Total parameters: {total_params:,}")

# Load checkpoint
print(f"\nLoading checkpoint from: {config['checkpoint']['load_path']}")
checkpoint_path = config['checkpoint']['load_path']
checkpoint = torch.load(checkpoint_path, map_location=device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
print(f"✓ Checkpoint loaded successfully")


## Step 5: Evaluate Model
선택한 데이터셋에서 모델을 평가합니다.


In [ ]:
print(f"Evaluating on {dataset_name} set...")
results = evaluate_model(model, eval_loader, lbl_min, lbl_max, device)

# Print evaluation summary
print_evaluation_summary(results)


## Step 6: Visualize Results
평가 결과를 4가지 플롯으로 시각화합니다.


In [ ]:
plot_results(results, save_path=config['evaluation']['save_results_plot'])
print(f"✓ Evaluation plot saved to {config['evaluation']['save_results_plot']}")


## Step 7: Save Predictions to CSV (Optional)
예측 결과를 CSV 파일로 저장합니다.


In [ ]:
# Save predictions to CSV
if config['evaluation'].get('save_predictions_csv'):
    csv_path = config['evaluation']['save_predictions_csv']
    df = pd.DataFrame({
        'true_values': results['true_vals'],
        'predicted_values': results['pred_vals'],
        'errors': results['errors'],
        'absolute_errors': results['abs_errors'],
        'absolute_relative_errors_percent': results['abs_rel_errors']
    })
    df.to_csv(csv_path, index=False)
    print(f"✓ Predictions saved to {csv_path}")
    
    # Display first few rows
    print("\nFirst 10 predictions:")
    print(df.head(10))


## Summary

평가가 완료되었습니다!

- **Evaluation plot**: `./results/evaluation_results.png`
- **Predictions CSV**: `./results/predictions.csv`

### 주요 메트릭:
- MSE, RMSE, MAE, MAPE
- Percentile 통계 (50th, 68th, 95th)
- 4가지 시각화 플롯
